<a href="https://colab.research.google.com/github/tousifo/ml_notebooks/blob/main/Bangladesh_Multi_Hazard_Dynamic_Simulator_All_In_One.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🇧🇩 Bangladesh Multi-Hazard Risk & Climate Vulnerability Simulator

This notebook builds a complete **dynamic Streamlit simulator** for Bangladesh multi-hazard risk.

It includes:

- Flood Risk Index simulation
- Cyclone Risk Index simulation
- Drought Risk Index simulation
- Agricultural Stress simulation
- Composite Vulnerability simulation
- What-if controls for rainfall, river level, temperature, AQI, forest cover, carbon emission, district, and scenario presets
- Model leaderboard with real R² scores
- District explorer, risk radar, national map, sensitivity analysis, and downloadable dataset workspace

**Important:** this project predicts transparent 0–10 risk-index scores. It does not claim to forecast exact real disaster events.



## 1. Install Required Packages

Run this once at the start of the Colab session.


In [ ]:

!pip -q install streamlit scikit-learn plotly joblib


ERROR: Operation cancelled by user



## 2. Check Dataset

Upload your dataset into Colab as:

`/content/Bangladesh_Environmental_Climate_Change_Impact.csv`


In [ ]:

from pathlib import Path
import pandas as pd

DATA_PATH = Path("/content/Bangladesh_Environmental_Climate_Change_Impact.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset not found. Please upload Bangladesh_Environmental_Climate_Change_Impact.csv to /content."
    )

df_preview = pd.read_csv(DATA_PATH)
print("✅ Dataset loaded")
print("Shape:", df_preview.shape)
display(df_preview.head())


✅ Dataset loaded
Shape: (3000, 15)


,Year,District,Avg_Temperature_C,Annual_Rainfall_mm,AQI,Forest_Cover_Percent,River_Water_Level_m,Cyclone_Count,Flood_Impact_Score,Drought_Severity,Agricultural_Yield_ton_per_hectare,Coastal_Erosion_m_per_year,Urbanization_Rate_Percent,Carbon_Emission_Metric_Tons_per_Capita,Renewable_Energy_Usage_Percent
0,2025,Satkhira,27.06,4044.6,257,9.36,5.89,1,6.01,9.25,5.21,1.15,54.15,1.20,18.20
1,2018,Tangail,25.86,1659.1,222,29.12,8.74,1,9.55,6.24,4.28,2.09,47.95,2.02,16.47
2,2013,Sunamganj,25.20,3618.1,56,24.01,8.45,1,7.47,4.83,2.90,4.41,65.13,1.77,11.23
3,2005,Sirajganj,27.85,4031.3,171,25.68,4.04,2,1.80,5.43,5.20,1.10,32.84,0.69,14.80
4,2013,Barishal,29.94,3844.4,259,8.13,9.83,3,0.10,9.89,4.65,4.17,65.95,1.33,19.98



## 3. Prepare Risk Indices + Train and Save Models

This cell creates the transparent risk-index targets and trains lightweight models once in the notebook.

The Streamlit app will load these saved models instead of training slowly every time the app opens.


In [ ]:

import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

DATA_PATH = Path("/content/Bangladesh_Environmental_Climate_Change_Impact.csv")
PROCESSED_PATH = Path("/content/multihazard_processed.csv")
MODELS_PATH = Path("/content/multihazard_models.joblib")
LEADERBOARD_PATH = Path("/content/multihazard_leaderboard.csv")
METADATA_PATH = Path("/content/multihazard_metadata.json")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Upload Bangladesh_Environmental_Climate_Change_Impact.csv to /content first."
    )

TARGET_MAP = {
    "🌊 Flood Risk": "Flood_Risk_Index",
    "🌪️ Cyclone Risk": "Cyclone_Risk_Index",
    "🔥 Drought Risk": "Drought_Risk_Index",
    "🌾 Agricultural Stress": "Agricultural_Stress_Index",
    "⚠️ Composite Vulnerability": "Composite_Vulnerability_Index"
}

FLOOD_DISTRICTS = [
    "Sunamganj", "Sylhet", "Kurigram", "Gaibandha",
    "Jamalpur", "Bogura", "Sirajganj", "Lalmonirhat",
    "Netrokona", "Sherpur"
]

CYCLONE_DISTRICTS = [
    "Bhola", "Patuakhali", "Satkhira", "Cox's Bazar",
    "Barguna", "Chittagong", "Chattogram", "Khulna",
    "Barisal", "Feni", "Noakhali", "Bagerhat"
]

DROUGHT_DISTRICTS = [
    "Rajshahi", "Naogaon", "Chapainawabganj",
    "Dinajpur", "Rangpur", "Kushtia",
    "Thakurgaon", "Meherpur", "Chuadanga"
]

COORDS = {
    "Bagerhat": [22.6602, 89.7895], "Bandarban": [22.1953, 92.2184],
    "Barguna": [22.1591, 90.1189], "Barisal": [22.7010, 90.3535],
    "Bhola": [22.6859, 90.6440], "Bogura": [24.8481, 89.3730],
    "Brahmanbaria": [23.9578, 91.1111], "Chandpur": [23.2333, 90.6333],
    "Chapainawabganj": [24.5965, 88.2775], "Chattogram": [22.3569, 91.7832],
    "Chittagong": [22.3569, 91.7832], "Chuadanga": [23.6402, 88.8418],
    "Comilla": [23.4682, 91.1788], "Cox's Bazar": [21.4272, 92.0058],
    "Dhaka": [23.8103, 90.4125], "Dinajpur": [25.6217, 88.6354],
    "Feni": [23.0159, 91.3976], "Gaibandha": [25.3288, 89.5280],
    "Gazipur": [24.0023, 90.4260], "Gopalganj": [23.0051, 89.8266],
    "Jamalpur": [24.9375, 89.9375], "Jashore": [23.1664, 89.2081],
    "Jessore": [23.1664, 89.2081], "Khulna": [22.8456, 89.5403],
    "Kurigram": [25.8103, 89.6487], "Kushtia": [23.9010, 89.1204],
    "Lalmonirhat": [25.9167, 89.4500], "Magura": [23.4855, 89.4198],
    "Manikganj": [23.8617, 90.0003], "Maulvibazar": [24.4829, 91.7774],
    "Meherpur": [23.7622, 88.6318], "Mymensingh": [24.7471, 90.4203],
    "Naogaon": [24.7936, 88.9318], "Narayanganj": [23.6238, 90.5000],
    "Narsingdi": [23.9322, 90.7154], "Netrokona": [24.8835, 90.7310],
    "Noakhali": [22.8698, 91.0991], "Pabna": [24.0064, 89.2372],
    "Patuakhali": [22.3597, 90.3297], "Rajshahi": [24.3745, 88.6042],
    "Rangpur": [25.7439, 89.2752], "Satkhira": [22.7185, 89.0705],
    "Sherpur": [25.0200, 90.0153], "Sirajganj": [24.4583, 89.7083],
    "Sunamganj": [25.0658, 91.3950], "Sylhet": [24.8949, 91.8687],
    "Thakurgaon": [26.0337, 88.4617]
}

NUMERIC_FEATURES = [
    "Year",
    "Avg_Temperature_C",
    "Annual_Rainfall_mm",
    "AQI",
    "Forest_Cover_Percent",
    "River_Water_Level_m",
    "Carbon_Emission_Metric_Tons_per_Capita",
    "Latitude",
    "Longitude",
    "Is_Flood_Prone_District",
    "Is_Cyclone_Prone_District",
    "Is_Drought_Prone_District",
    "Rainfall_Norm",
    "River_Norm",
    "Temperature_Norm",
    "AQI_Norm",
    "Carbon_Norm",
    "Forest_Loss_Norm"
]

CATEGORICAL_FEATURES = ["District"]
FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

required_cols = [
    "District", "Year", "Avg_Temperature_C", "Annual_Rainfall_mm", "AQI",
    "Forest_Cover_Percent", "River_Water_Level_m",
    "Carbon_Emission_Metric_Tons_per_Capita"
]

raw = pd.read_csv(DATA_PATH)
missing = [c for c in required_cols if c not in raw.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

def safe_minmax(series):
    denominator = series.max() - series.min()
    if denominator == 0:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - series.min()) / denominator

def add_risk_indices(raw_df):
    df = raw_df.copy()

    df["Latitude"] = df["District"].map(lambda x: COORDS.get(x, [23.6850, 90.3563])[0])
    df["Longitude"] = df["District"].map(lambda x: COORDS.get(x, [23.6850, 90.3563])[1])

    df["Is_Flood_Prone_District"] = df["District"].isin(FLOOD_DISTRICTS).astype(int)
    df["Is_Cyclone_Prone_District"] = df["District"].isin(CYCLONE_DISTRICTS).astype(int)
    df["Is_Drought_Prone_District"] = df["District"].isin(DROUGHT_DISTRICTS).astype(int)

    df["Rainfall_Norm"] = safe_minmax(df["Annual_Rainfall_mm"])
    df["River_Norm"] = safe_minmax(df["River_Water_Level_m"])
    df["Temperature_Norm"] = safe_minmax(df["Avg_Temperature_C"])
    df["AQI_Norm"] = safe_minmax(df["AQI"])
    df["Carbon_Norm"] = safe_minmax(df["Carbon_Emission_Metric_Tons_per_Capita"])
    df["Forest_Loss_Norm"] = 1 - safe_minmax(df["Forest_Cover_Percent"])

    df["Flood_Risk_Index"] = (
        0.35 * df["Rainfall_Norm"] +
        0.30 * df["River_Norm"] +
        0.15 * df["Is_Flood_Prone_District"] +
        0.10 * df["Forest_Loss_Norm"] +
        0.05 * df["Temperature_Norm"] +
        0.05 * df["Carbon_Norm"]
    ) * 10

    df["Cyclone_Risk_Index"] = (
        0.35 * df["Is_Cyclone_Prone_District"] +
        0.20 * df["Rainfall_Norm"] +
        0.15 * df["Temperature_Norm"] +
        0.10 * df["River_Norm"] +
        0.10 * df["Carbon_Norm"] +
        0.10 * df["Forest_Loss_Norm"]
    ) * 10

    df["Drought_Risk_Index"] = (
        0.30 * df["Is_Drought_Prone_District"] +
        0.25 * df["Temperature_Norm"] +
        0.20 * (1 - df["Rainfall_Norm"]) +
        0.10 * (1 - df["River_Norm"]) +
        0.10 * df["Forest_Loss_Norm"] +
        0.05 * df["Carbon_Norm"]
    ) * 10

    df["Agricultural_Stress_Index"] = (
        0.30 * df["Drought_Risk_Index"] / 10 +
        0.20 * df["Flood_Risk_Index"] / 10 +
        0.15 * df["Temperature_Norm"] +
        0.15 * df["Forest_Loss_Norm"] +
        0.10 * df["AQI_Norm"] +
        0.10 * df["Carbon_Norm"]
    ) * 10

    df["Composite_Vulnerability_Index"] = (
        0.30 * df["Flood_Risk_Index"] +
        0.25 * df["Cyclone_Risk_Index"] +
        0.25 * df["Drought_Risk_Index"] +
        0.20 * df["Agricultural_Stress_Index"]
    )

    return df

def make_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def build_models(data):
    X = data[FEATURE_COLUMNS]

    model_candidates = {
        "Extra Trees": ExtraTreesRegressor(n_estimators=120, max_depth=10, random_state=42, n_jobs=-1),
        "Random Forest": RandomForestRegressor(n_estimators=120, max_depth=10, random_state=42, n_jobs=-1),
        "Gradient Boosting": GradientBoostingRegressor(n_estimators=120, learning_rate=0.05, max_depth=3, random_state=42),
    }

    all_results = []
    best_models = {}

    for hazard_label, target_col in TARGET_MAP.items():
        y = data[target_col]

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.20, random_state=42
        )

        preprocessor = ColumnTransformer(
            transformers=[
                ("num", StandardScaler(), NUMERIC_FEATURES),
                ("cat", make_encoder(), CATEGORICAL_FEATURES),
            ],
            sparse_threshold=0
        )

        fitted = {}

        for model_name, regressor in model_candidates.items():
            pipe = Pipeline([
                ("preprocessor", preprocessor),
                ("model", regressor),
            ])

            pipe.fit(X_train, y_train)

            train_pred = pipe.predict(X_train)
            test_pred = pipe.predict(X_test)

            row = {
                "Hazard": hazard_label,
                "Target": target_col,
                "Model": model_name,
                "Train_R2": r2_score(y_train, train_pred),
                "Test_R2": r2_score(y_test, test_pred),
                "MAE": mean_absolute_error(y_test, test_pred),
                "RMSE": float(np.sqrt(mean_squared_error(y_test, test_pred))),
            }

            all_results.append(row)
            fitted[model_name] = pipe

        target_rows = pd.DataFrame([r for r in all_results if r["Target"] == target_col])
        best_model_name = target_rows.sort_values("Test_R2", ascending=False).iloc[0]["Model"]
        best_models[hazard_label] = fitted[best_model_name]

    return best_models, pd.DataFrame(all_results)

df_model = add_risk_indices(raw)

best_models, leaderboard = build_models(df_model)

df_model.to_csv(PROCESSED_PATH, index=False)
leaderboard.to_csv(LEADERBOARD_PATH, index=False)

joblib.dump({
    "best_models": best_models,
    "target_map": TARGET_MAP,
    "feature_columns": FEATURE_COLUMNS,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
}, MODELS_PATH)

metadata = {
    "flood_districts": FLOOD_DISTRICTS,
    "cyclone_districts": CYCLONE_DISTRICTS,
    "drought_districts": DROUGHT_DISTRICTS,
    "coords": COORDS,
}
METADATA_PATH.write_text(json.dumps(metadata, indent=2))

print("✅ Processed dataset saved:", PROCESSED_PATH)
print("✅ Best models saved:", MODELS_PATH)
print("✅ Leaderboard saved:", LEADERBOARD_PATH)
print("✅ Metadata saved:", METADATA_PATH)

best_summary = leaderboard.sort_values("Test_R2", ascending=False).groupby("Hazard").head(1)
display(best_summary.sort_values("Hazard"))


✅ Processed dataset saved: /content/multihazard_processed.csv
✅ Best models saved: /content/multihazard_models.joblib
✅ Leaderboard saved: /content/multihazard_leaderboard.csv
✅ Metadata saved: /content/multihazard_metadata.json


,Hazard,Target,Model,Train_R2,Test_R2,MAE,RMSE
12,⚠️ Composite Vulnerability,Composite_Vulnerability_Index,Extra Trees,0.983192,0.933099,0.173085,0.218002
2,🌊 Flood Risk,Flood_Risk_Index,Gradient Boosting,0.989801,0.985281,0.146900,0.186816
5,🌪️ Cyclone Risk,Cyclone_Risk_Index,Gradient Boosting,0.990778,0.985034,0.144051,0.182210
11,🌾 Agricultural Stress,Agricultural_Stress_Index,Gradient Boosting,0.980873,0.964916,0.156741,0.196696
8,🔥 Drought Risk,Drought_Risk_Index,Gradient Boosting,0.990429,0.984176,0.144066,0.180318



## 4. Write the Dynamic Streamlit App

This cell writes the full app to `/content/app.py`.

The app includes:

- Executive dashboard
- What-if simulator
- Scenario comparison lab
- Multi-hazard radar
- Model leaderboard
- District explorer
- Dataset workspace


In [ ]:
%%writefile /content/app.py
import os
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import streamlit as st
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# ============================================================
# APP CONFIG
# ============================================================

st.set_page_config(
    page_title="Bangladesh Multi-Hazard AI Simulator",
    page_icon="🇧🇩",
    layout="wide",
    initial_sidebar_state="expanded"
)

DATA_PATH = "/content/Bangladesh_Environmental_Climate_Change_Impact.csv"
PROCESSED_PATH = "/content/multihazard_processed.csv"
MODELS_PATH = "/content/multihazard_models.joblib"
LEADERBOARD_PATH = "/content/multihazard_leaderboard.csv"
METADATA_PATH = "/content/multihazard_metadata.json"

TARGET_MAP = {
    "🌊 Flood Risk": "Flood_Risk_Index",
    "🌪️ Cyclone Risk": "Cyclone_Risk_Index",
    "🔥 Drought Risk": "Drought_Risk_Index",
    "🌾 Agricultural Stress": "Agricultural_Stress_Index",
    "⚠️ Composite Vulnerability": "Composite_Vulnerability_Index"
}

RISK_COLUMNS = list(TARGET_MAP.values())

# ============================================================
# BEAUTIFUL UI CSS
# ============================================================

st.markdown("""
<style>
:root {
    --bg: #07111f;
    --panel: rgba(15, 23, 42, 0.88);
    --panel2: rgba(30, 41, 59, 0.92);
    --text: #f8fafc;
    --muted: #cbd5e1;
    --blue: #38bdf8;
    --green: #22c55e;
    --yellow: #facc15;
    --orange: #fb923c;
    --red: #ef4444;
}
.stApp {
    background: radial-gradient(circle at top left, #1e3a8a 0%, #0f172a 32%, #020617 100%);
    color: var(--text);
}
[data-testid="stSidebar"] {
    background: linear-gradient(180deg, #020617 0%, #0f172a 100%);
    border-right: 1px solid rgba(148, 163, 184, 0.22);
}
.block-container {
    padding-top: 1.5rem;
}
.main-title {
    font-size: 42px;
    font-weight: 900;
    color: white;
    line-height: 1.12;
    letter-spacing: -0.02em;
}
.sub-title {
    color: #cbd5e1;
    font-size: 17px;
    margin-top: 8px;
    margin-bottom: 18px;
}
.glass-card {
    background: rgba(15, 23, 42, 0.82);
    padding: 22px;
    border-radius: 22px;
    border: 1px solid rgba(148, 163, 184, 0.22);
    box-shadow: 0 18px 42px rgba(0,0,0,0.30);
}
.info-card {
    background: rgba(30, 41, 59, 0.85);
    padding: 20px 22px;
    border-radius: 18px;
    border-left: 6px solid #38bdf8;
    color: #e2e8f0;
    margin-bottom: 20px;
}
.warning-card {
    background: rgba(120, 53, 15, 0.75);
    padding: 18px 20px;
    border-radius: 18px;
    border-left: 6px solid #facc15;
    color: #fef3c7;
}
.kpi-title {
    color: #cbd5e1;
    font-size: 14px;
    margin-bottom: 8px;
}
.kpi-number {
    font-size: 38px;
    font-weight: 900;
    color: #38bdf8;
}
.kpi-small {
    color: #94a3b8;
    font-size: 13px;
}
.risk-low { color: #22c55e; font-weight: 900; }
.risk-moderate { color: #facc15; font-weight: 900; }
.risk-high { color: #fb923c; font-weight: 900; }
.risk-very-high { color: #ef4444; font-weight: 900; }
hr {
    border: none;
    border-top: 1px solid rgba(148, 163, 184, 0.25);
    margin: 1.4rem 0;
}
div[data-testid="stMetric"] {
    background: rgba(15,23,42,.70);
    border: 1px solid rgba(148,163,184,.22);
    padding: 18px;
    border-radius: 18px;
}
</style>
""", unsafe_allow_html=True)

# ============================================================
# CONSTANTS AND FEATURE ENGINEERING
# ============================================================

FLOOD_DISTRICTS = [
    "Sunamganj", "Sylhet", "Kurigram", "Gaibandha",
    "Jamalpur", "Bogura", "Sirajganj", "Lalmonirhat",
    "Netrokona", "Sherpur"
]

CYCLONE_DISTRICTS = [
    "Bhola", "Patuakhali", "Satkhira", "Cox's Bazar",
    "Barguna", "Chittagong", "Chattogram", "Khulna",
    "Barisal", "Feni", "Noakhali", "Bagerhat"
]

DROUGHT_DISTRICTS = [
    "Rajshahi", "Naogaon", "Chapainawabganj",
    "Dinajpur", "Rangpur", "Kushtia",
    "Thakurgaon", "Meherpur", "Chuadanga"
]

COORDS = {
    "Bagerhat": [22.6602, 89.7895], "Bandarban": [22.1953, 92.2184],
    "Barguna": [22.1591, 90.1189], "Barisal": [22.7010, 90.3535],
    "Bhola": [22.6859, 90.6440], "Bogura": [24.8481, 89.3730],
    "Brahmanbaria": [23.9578, 91.1111], "Chandpur": [23.2333, 90.6333],
    "Chapainawabganj": [24.5965, 88.2775], "Chattogram": [22.3569, 91.7832],
    "Chittagong": [22.3569, 91.7832], "Chuadanga": [23.6402, 88.8418],
    "Comilla": [23.4682, 91.1788], "Cox's Bazar": [21.4272, 92.0058],
    "Dhaka": [23.8103, 90.4125], "Dinajpur": [25.6217, 88.6354],
    "Feni": [23.0159, 91.3976], "Gaibandha": [25.3288, 89.5280],
    "Gazipur": [24.0023, 90.4260], "Gopalganj": [23.0051, 89.8266],
    "Jamalpur": [24.9375, 89.9375], "Jashore": [23.1664, 89.2081],
    "Jessore": [23.1664, 89.2081], "Khulna": [22.8456, 89.5403],
    "Kurigram": [25.8103, 89.6487], "Kushtia": [23.9010, 89.1204],
    "Lalmonirhat": [25.9167, 89.4500], "Magura": [23.4855, 89.4198],
    "Manikganj": [23.8617, 90.0003], "Maulvibazar": [24.4829, 91.7774],
    "Meherpur": [23.7622, 88.6318], "Mymensingh": [24.7471, 90.4203],
    "Naogaon": [24.7936, 88.9318], "Narayanganj": [23.6238, 90.5000],
    "Narsingdi": [23.9322, 90.7154], "Netrokona": [24.8835, 90.7310],
    "Noakhali": [22.8698, 91.0991], "Pabna": [24.0064, 89.2372],
    "Patuakhali": [22.3597, 90.3297], "Rajshahi": [24.3745, 88.6042],
    "Rangpur": [25.7439, 89.2752], "Satkhira": [22.7185, 89.0705],
    "Sherpur": [25.0200, 90.0153], "Sirajganj": [24.4583, 89.7083],
    "Sunamganj": [25.0658, 91.3950], "Sylhet": [24.8949, 91.8687],
    "Thakurgaon": [26.0337, 88.4617]
}

NUMERIC_FEATURES = [
    "Year",
    "Avg_Temperature_C",
    "Annual_Rainfall_mm",
    "AQI",
    "Forest_Cover_Percent",
    "River_Water_Level_m",
    "Carbon_Emission_Metric_Tons_per_Capita",
    "Latitude",
    "Longitude",
    "Is_Flood_Prone_District",
    "Is_Cyclone_Prone_District",
    "Is_Drought_Prone_District",
    "Rainfall_Norm",
    "River_Norm",
    "Temperature_Norm",
    "AQI_Norm",
    "Carbon_Norm",
    "Forest_Loss_Norm"
]

CATEGORICAL_FEATURES = ["District"]
FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES


def safe_minmax(series: pd.Series) -> pd.Series:
    denom = series.max() - series.min()
    if denom == 0:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - series.min()) / denom


def add_risk_indices(raw_df: pd.DataFrame) -> pd.DataFrame:
    df = raw_df.copy()

    df["Latitude"] = df["District"].map(lambda x: COORDS.get(x, [23.6850, 90.3563])[0])
    df["Longitude"] = df["District"].map(lambda x: COORDS.get(x, [23.6850, 90.3563])[1])

    df["Is_Flood_Prone_District"] = df["District"].isin(FLOOD_DISTRICTS).astype(int)
    df["Is_Cyclone_Prone_District"] = df["District"].isin(CYCLONE_DISTRICTS).astype(int)
    df["Is_Drought_Prone_District"] = df["District"].isin(DROUGHT_DISTRICTS).astype(int)

    df["Rainfall_Norm"] = safe_minmax(df["Annual_Rainfall_mm"])
    df["River_Norm"] = safe_minmax(df["River_Water_Level_m"])
    df["Temperature_Norm"] = safe_minmax(df["Avg_Temperature_C"])
    df["AQI_Norm"] = safe_minmax(df["AQI"])
    df["Carbon_Norm"] = safe_minmax(df["Carbon_Emission_Metric_Tons_per_Capita"])
    df["Forest_Loss_Norm"] = 1 - safe_minmax(df["Forest_Cover_Percent"])

    df["Flood_Risk_Index"] = (
        0.35 * df["Rainfall_Norm"] +
        0.30 * df["River_Norm"] +
        0.15 * df["Is_Flood_Prone_District"] +
        0.10 * df["Forest_Loss_Norm"] +
        0.05 * df["Temperature_Norm"] +
        0.05 * df["Carbon_Norm"]
    ) * 10

    df["Cyclone_Risk_Index"] = (
        0.35 * df["Is_Cyclone_Prone_District"] +
        0.20 * df["Rainfall_Norm"] +
        0.15 * df["Temperature_Norm"] +
        0.10 * df["River_Norm"] +
        0.10 * df["Carbon_Norm"] +
        0.10 * df["Forest_Loss_Norm"]
    ) * 10

    df["Drought_Risk_Index"] = (
        0.30 * df["Is_Drought_Prone_District"] +
        0.25 * df["Temperature_Norm"] +
        0.20 * (1 - df["Rainfall_Norm"]) +
        0.10 * (1 - df["River_Norm"]) +
        0.10 * df["Forest_Loss_Norm"] +
        0.05 * df["Carbon_Norm"]
    ) * 10

    df["Agricultural_Stress_Index"] = (
        0.30 * df["Drought_Risk_Index"] / 10 +
        0.20 * df["Flood_Risk_Index"] / 10 +
        0.15 * df["Temperature_Norm"] +
        0.15 * df["Forest_Loss_Norm"] +
        0.10 * df["AQI_Norm"] +
        0.10 * df["Carbon_Norm"]
    ) * 10

    df["Composite_Vulnerability_Index"] = (
        0.30 * df["Flood_Risk_Index"] +
        0.25 * df["Cyclone_Risk_Index"] +
        0.25 * df["Drought_Risk_Index"] +
        0.20 * df["Agricultural_Stress_Index"]
    )

    return df


def make_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_models(data: pd.DataFrame):
    X = data[FEATURE_COLUMNS]
    all_results = []
    best_models = {}

    model_candidates = {
        "Extra Trees": ExtraTreesRegressor(n_estimators=120, max_depth=10, random_state=42, n_jobs=-1),
        "Random Forest": RandomForestRegressor(n_estimators=120, max_depth=10, random_state=42, n_jobs=-1),
        "Gradient Boosting": GradientBoostingRegressor(n_estimators=120, learning_rate=0.05, max_depth=3, random_state=42),
    }

    for hazard_label, target_col in TARGET_MAP.items():
        y = data[target_col]

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

        preprocessor = ColumnTransformer(
            transformers=[
                ("num", StandardScaler(), NUMERIC_FEATURES),
                ("cat", make_encoder(), CATEGORICAL_FEATURES)
            ],
            sparse_threshold=0
        )

        fitted = {}

        for model_name, regressor in model_candidates.items():
            model = Pipeline([
                ("preprocessor", preprocessor),
                ("model", regressor)
            ])

            model.fit(X_train, y_train)

            train_pred = model.predict(X_train)
            test_pred = model.predict(X_test)

            row = {
                "Hazard": hazard_label,
                "Target": target_col,
                "Model": model_name,
                "Train_R2": r2_score(y_train, train_pred),
                "Test_R2": r2_score(y_test, test_pred),
                "MAE": mean_absolute_error(y_test, test_pred),
                "RMSE": float(np.sqrt(mean_squared_error(y_test, test_pred)))
            }

            all_results.append(row)
            fitted[model_name] = model

        target_rows = pd.DataFrame([r for r in all_results if r["Target"] == target_col])
        best_model_name = target_rows.sort_values("Test_R2", ascending=False).iloc[0]["Model"]
        best_models[hazard_label] = fitted[best_model_name]

    return best_models, pd.DataFrame(all_results)


@st.cache_data(show_spinner=False)
def load_processed_data():
    if Path(PROCESSED_PATH).exists():
        return pd.read_csv(PROCESSED_PATH)
    if not Path(DATA_PATH).exists():
        st.error("Dataset not found. Upload Bangladesh_Environmental_Climate_Change_Impact.csv into /content first.")
        st.stop()
    raw = pd.read_csv(DATA_PATH)
    return add_risk_indices(raw)


@st.cache_resource(show_spinner=False)
def load_or_train_models(data):
    if Path(MODELS_PATH).exists() and Path(LEADERBOARD_PATH).exists():
        package = joblib.load(MODELS_PATH)
        leaderboard_df = pd.read_csv(LEADERBOARD_PATH)
        return package["best_models"], leaderboard_df

    with st.spinner("Training lightweight models because saved model files were not found..."):
        models, leaderboard_df = build_models(data)
    return models, leaderboard_df


df = load_processed_data()
best_models, leaderboard = load_or_train_models(df)

# ============================================================
# SIMULATION HELPERS
# ============================================================

def norm_from_dataset(value, column):
    min_v = float(df[column].min())
    max_v = float(df[column].max())
    if max_v - min_v == 0:
        return 0.0
    return float(np.clip((value - min_v) / (max_v - min_v), 0, 1))


def make_input_row(district, year, temperature, rainfall, aqi, forest, river, carbon):
    lat = COORDS.get(district, [23.6850, 90.3563])[0]
    lon = COORDS.get(district, [23.6850, 90.3563])[1]

    rainfall_norm = norm_from_dataset(rainfall, "Annual_Rainfall_mm")
    river_norm = norm_from_dataset(river, "River_Water_Level_m")
    temp_norm = norm_from_dataset(temperature, "Avg_Temperature_C")
    aqi_norm = norm_from_dataset(aqi, "AQI")
    carbon_norm = norm_from_dataset(carbon, "Carbon_Emission_Metric_Tons_per_Capita")
    forest_loss_norm = 1 - norm_from_dataset(forest, "Forest_Cover_Percent")

    return pd.DataFrame([{
        "Year": year,
        "Avg_Temperature_C": temperature,
        "Annual_Rainfall_mm": rainfall,
        "AQI": aqi,
        "Forest_Cover_Percent": forest,
        "River_Water_Level_m": river,
        "Carbon_Emission_Metric_Tons_per_Capita": carbon,
        "Latitude": lat,
        "Longitude": lon,
        "Is_Flood_Prone_District": 1 if district in FLOOD_DISTRICTS else 0,
        "Is_Cyclone_Prone_District": 1 if district in CYCLONE_DISTRICTS else 0,
        "Is_Drought_Prone_District": 1 if district in DROUGHT_DISTRICTS else 0,
        "Rainfall_Norm": rainfall_norm,
        "River_Norm": river_norm,
        "Temperature_Norm": temp_norm,
        "AQI_Norm": aqi_norm,
        "Carbon_Norm": carbon_norm,
        "Forest_Loss_Norm": forest_loss_norm,
        "District": district
    }])


def predict_all(input_row):
    predictions = {}
    for label, model in best_models.items():
        score = float(model.predict(input_row)[0])
        predictions[label] = round(max(0, min(10, score)), 3)
    return predictions


def risk_label(score):
    if score >= 7.5:
        return "Very High", "risk-very-high"
    if score >= 5.5:
        return "High", "risk-high"
    if score >= 3.5:
        return "Moderate", "risk-moderate"
    return "Low", "risk-low"


def components_for_hazard(hazard_label, input_row, predictions):
    r = input_row.iloc[0]
    if hazard_label == "🌊 Flood Risk":
        return pd.DataFrame({
            "Factor": ["Rainfall", "River level", "Flood-prone district", "Forest loss", "Temperature", "Carbon"],
            "Contribution": [
                3.5*r["Rainfall_Norm"],
                3.0*r["River_Norm"],
                1.5*r["Is_Flood_Prone_District"],
                1.0*r["Forest_Loss_Norm"],
                0.5*r["Temperature_Norm"],
                0.5*r["Carbon_Norm"],
            ]
        })
    if hazard_label == "🌪️ Cyclone Risk":
        return pd.DataFrame({
            "Factor": ["Cyclone-prone district", "Rainfall", "Temperature", "River level", "Carbon", "Forest loss"],
            "Contribution": [
                3.5*r["Is_Cyclone_Prone_District"],
                2.0*r["Rainfall_Norm"],
                1.5*r["Temperature_Norm"],
                1.0*r["River_Norm"],
                1.0*r["Carbon_Norm"],
                1.0*r["Forest_Loss_Norm"],
            ]
        })
    if hazard_label == "🔥 Drought Risk":
        return pd.DataFrame({
            "Factor": ["Drought-prone district", "Temperature", "Low rainfall", "Low river level", "Forest loss", "Carbon"],
            "Contribution": [
                3.0*r["Is_Drought_Prone_District"],
                2.5*r["Temperature_Norm"],
                2.0*(1-r["Rainfall_Norm"]),
                1.0*(1-r["River_Norm"]),
                1.0*r["Forest_Loss_Norm"],
                0.5*r["Carbon_Norm"],
            ]
        })
    if hazard_label == "🌾 Agricultural Stress":
        return pd.DataFrame({
            "Factor": ["Drought risk", "Flood risk", "Temperature", "Forest loss", "AQI", "Carbon"],
            "Contribution": [
                3.0*predictions["🔥 Drought Risk"]/10,
                2.0*predictions["🌊 Flood Risk"]/10,
                1.5*r["Temperature_Norm"],
                1.5*r["Forest_Loss_Norm"],
                1.0*r["AQI_Norm"],
                1.0*r["Carbon_Norm"],
            ]
        })
    return pd.DataFrame({
        "Factor": ["Flood risk", "Cyclone risk", "Drought risk", "Agricultural stress"],
        "Contribution": [
            0.30*predictions["🌊 Flood Risk"],
            0.25*predictions["🌪️ Cyclone Risk"],
            0.25*predictions["🔥 Drought Risk"],
            0.20*predictions["🌾 Agricultural Stress"],
        ]
    })


def apply_preset(preset, values):
    year, temp, rainfall, aqi, forest, river, carbon = values

    if preset == "Heavy rainfall / flood stress":
        rainfall = min(5000, int(rainfall * 1.40))
        river = min(20.0, river * 1.35)
    elif preset == "Cyclone-prone coastal stress":
        rainfall = min(5000, int(rainfall * 1.25))
        river = min(20.0, river * 1.20)
        carbon = min(5.0, carbon + 0.40)
    elif preset == "Extreme drought / heatwave":
        temp = min(45.0, temp + 4.0)
        rainfall = max(500, int(rainfall * 0.50))
        river = max(0.0, river * 0.55)
        forest = max(1.0, forest - 4.0)
    elif preset == "Environmental improvement":
        forest = min(50.0, forest + 10.0)
        carbon = max(0.1, carbon - 0.45)
        aqi = max(10, int(aqi * 0.70))
    elif preset == "Worst-case compound hazard":
        temp = min(45.0, temp + 3.0)
        rainfall = min(5000, int(rainfall * 1.30))
        river = min(20.0, river * 1.25)
        forest = max(1.0, forest - 8.0)
        carbon = min(5.0, carbon + 0.60)
        aqi = min(300, int(aqi * 1.25))

    return year, temp, rainfall, aqi, forest, river, carbon


# ============================================================
# SIDEBAR
# ============================================================

st.sidebar.markdown("## ⚙️ Control Panel")
page = st.sidebar.radio(
    "Navigation Menu",
    [
        "🏠 Executive Dashboard",
        "🧪 What-if Simulator",
        "📈 Scenario Comparison",
        "🗺️ Multi-Hazard Risk Radar",
        "🤖 Model Leaderboard",
        "📍 District Explorer",
        "📊 Dataset Workspace"
    ]
)

st.sidebar.markdown("---")
st.sidebar.write(f"Latest dataset year: **{int(df['Year'].max())}**")
st.sidebar.write(f"Records loaded: **{len(df):,}**")
st.sidebar.write(f"Districts: **{df['District'].nunique()}**")

st.markdown('<div class="main-title">🇧🇩 Bangladesh Multi-Hazard Risk & Climate Vulnerability Simulator</div>', unsafe_allow_html=True)
st.markdown('<div class="sub-title">Dynamic what-if modelling for flood, cyclone, drought, agricultural stress, and composite vulnerability.</div>', unsafe_allow_html=True)
st.markdown("---")

latest_df = df[df["Year"] == df["Year"].max()].copy()

# ============================================================
# PAGE: DASHBOARD
# ============================================================

if page == "🏠 Executive Dashboard":
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Avg Flood Risk", f"{latest_df['Flood_Risk_Index'].mean():.2f}/10")
    c2.metric("Avg Cyclone Risk", f"{latest_df['Cyclone_Risk_Index'].mean():.2f}/10")
    c3.metric("Avg Drought Risk", f"{latest_df['Drought_Risk_Index'].mean():.2f}/10")
    c4.metric("Avg Composite Risk", f"{latest_df['Composite_Vulnerability_Index'].mean():.2f}/10")

    st.markdown("### National Multi-Hazard Map")
    fig = px.scatter_mapbox(
        latest_df,
        lat="Latitude",
        lon="Longitude",
        color="Composite_Vulnerability_Index",
        size="Composite_Vulnerability_Index",
        hover_name="District",
        hover_data=RISK_COLUMNS,
        zoom=5.5,
        height=540,
        color_continuous_scale="Turbo"
    )
    fig.update_layout(mapbox_style="carto-darkmatter", margin={"r":0, "t":0, "l":0, "b":0})
    st.plotly_chart(fig, use_container_width=True)

    st.markdown("### Average Risk by Hazard")
    avg_df = latest_df[RISK_COLUMNS].mean().reset_index()
    avg_df.columns = ["Risk Type", "Average Score"]
    fig_avg = px.bar(avg_df, x="Risk Type", y="Average Score", color="Average Score", color_continuous_scale="Turbo")
    st.plotly_chart(fig_avg, use_container_width=True)

# ============================================================
# PAGE: WHAT-IF SIMULATOR
# ============================================================

elif page == "🧪 What-if Simulator":
    st.markdown("## 🧪 Dynamic What-if Risk Simulator")

    st.markdown("""
    <div class="info-card">
    Change rainfall, river level, temperature, AQI, forest cover, carbon emission, district, and scenario preset.
    The simulator predicts all hazard scores and shows how the scenario changes risk compared with the district baseline.
    </div>
    """, unsafe_allow_html=True)

    col_a, col_b = st.columns([1.15, 0.85])

    with col_a:
        district = st.selectbox("Choose district", sorted(df["District"].unique()))
        hazard_choice = st.selectbox("Primary hazard to analyse", list(TARGET_MAP.keys()))

    district_latest = df[df["District"] == district].sort_values("Year").iloc[-1]

    with col_b:
        preset = st.selectbox(
            "Quick scenario preset",
            [
                "Custom",
                "Heavy rainfall / flood stress",
                "Cyclone-prone coastal stress",
                "Extreme drought / heatwave",
                "Environmental improvement",
                "Worst-case compound hazard"
            ]
        )

    base_values = (
        int(district_latest["Year"]),
        float(district_latest["Avg_Temperature_C"]),
        int(district_latest["Annual_Rainfall_mm"]),
        int(district_latest["AQI"]),
        float(district_latest["Forest_Cover_Percent"]),
        float(district_latest["River_Water_Level_m"]),
        float(district_latest["Carbon_Emission_Metric_Tons_per_Capita"])
    )

    preset_values = apply_preset(preset, base_values)

    c1, c2, c3 = st.columns(3)
    with c1:
        year = st.slider("Year", int(df["Year"].min()), int(df["Year"].max()), int(preset_values[0]))
        temperature = st.slider("Average temperature (°C)", 15.0, 45.0, float(preset_values[1]))
        rainfall = st.slider("Annual rainfall (mm)", 500, 5000, int(preset_values[2]))
    with c2:
        river = st.slider("River water level (m)", 0.0, 20.0, float(preset_values[5]))
        forest = st.slider("Forest cover (%)", 1.0, 50.0, float(preset_values[4]))
        aqi = st.slider("Air Quality Index", 10, 300, int(preset_values[3]))
    with c3:
        carbon = st.slider("Carbon emission metric tons per capita", 0.1, 5.0, float(preset_values[6]))
        show_components = st.toggle("Show risk-factor contribution", value=True)
        show_input = st.toggle("Show model input row", value=False)

    scenario_row = make_input_row(district, year, temperature, rainfall, aqi, forest, river, carbon)
    baseline_row = make_input_row(
        district,
        int(district_latest["Year"]),
        float(district_latest["Avg_Temperature_C"]),
        int(district_latest["Annual_Rainfall_mm"]),
        int(district_latest["AQI"]),
        float(district_latest["Forest_Cover_Percent"]),
        float(district_latest["River_Water_Level_m"]),
        float(district_latest["Carbon_Emission_Metric_Tons_per_Capita"])
    )

    scenario_preds = predict_all(scenario_row)
    baseline_preds = predict_all(baseline_row)

    selected_score = scenario_preds[hazard_choice]
    selected_label, selected_class = risk_label(selected_score)

    st.markdown("---")
    k1, k2, k3 = st.columns(3)
    with k1:
        st.markdown(f"""
        <div class="glass-card">
            <div class="kpi-title">Selected hazard score</div>
            <div class="kpi-number">{selected_score:.2f}/10</div>
            <div class="kpi-small">{hazard_choice}</div>
        </div>
        """, unsafe_allow_html=True)
    with k2:
        st.markdown(f"""
        <div class="glass-card">
            <div class="kpi-title">Risk category</div>
            <h1 class="{selected_class}">{selected_label}</h1>
            <div class="kpi-small">Scenario classification</div>
        </div>
        """, unsafe_allow_html=True)
    with k3:
        baseline_selected = baseline_preds[hazard_choice]
        delta = selected_score - baseline_selected
        st.metric("Change vs district baseline", f"{delta:+.2f}", f"Baseline: {baseline_selected:.2f}/10")

    st.markdown("### All Hazard Scores")
    score_df = pd.DataFrame({
        "Hazard": list(scenario_preds.keys()),
        "Baseline": [baseline_preds[k] for k in scenario_preds.keys()],
        "Scenario": [scenario_preds[k] for k in scenario_preds.keys()]
    })
    score_df["Change"] = score_df["Scenario"] - score_df["Baseline"]

    fig_scores = go.Figure()
    fig_scores.add_trace(go.Bar(x=score_df["Hazard"], y=score_df["Baseline"], name="Baseline"))
    fig_scores.add_trace(go.Bar(x=score_df["Hazard"], y=score_df["Scenario"], name="Scenario"))
    fig_scores.update_layout(barmode="group", yaxis_title="Risk score / 10", height=420)
    st.plotly_chart(fig_scores, use_container_width=True)

    if show_components:
        st.markdown("### Why the score changed")
        comp_df = components_for_hazard(hazard_choice, scenario_row, scenario_preds)
        comp_df = comp_df.sort_values("Contribution", ascending=True)
        fig_comp = px.bar(
            comp_df,
            x="Contribution",
            y="Factor",
            orientation="h",
            color="Contribution",
            color_continuous_scale="Turbo",
            title=f"Approximate factor contribution — {hazard_choice}"
        )
        st.plotly_chart(fig_comp, use_container_width=True)

    st.markdown("### Sensitivity Analysis")
    sens_col1, sens_col2 = st.columns([0.8, 1.2])
    with sens_col1:
        sensitivity_variable = st.selectbox(
            "Variable to stress-test",
            ["Rainfall", "River water level", "Temperature", "Forest cover", "AQI", "Carbon emission"]
        )
    with sens_col2:
        st.write("This chart changes one variable across a range while keeping the other scenario inputs fixed.")

    variable_ranges = {
        "Rainfall": np.linspace(500, 5000, 18),
        "River water level": np.linspace(0, 20, 18),
        "Temperature": np.linspace(15, 45, 18),
        "Forest cover": np.linspace(1, 50, 18),
        "AQI": np.linspace(10, 300, 18),
        "Carbon emission": np.linspace(0.1, 5.0, 18),
    }

    rows = []
    for val in variable_ranges[sensitivity_variable]:
        t_temperature, t_rainfall, t_aqi, t_forest, t_river, t_carbon = temperature, rainfall, aqi, forest, river, carbon
        if sensitivity_variable == "Rainfall":
            t_rainfall = int(val)
        elif sensitivity_variable == "River water level":
            t_river = float(val)
        elif sensitivity_variable == "Temperature":
            t_temperature = float(val)
        elif sensitivity_variable == "Forest cover":
            t_forest = float(val)
        elif sensitivity_variable == "AQI":
            t_aqi = int(val)
        elif sensitivity_variable == "Carbon emission":
            t_carbon = float(val)

        temp_row = make_input_row(district, year, t_temperature, t_rainfall, t_aqi, t_forest, t_river, t_carbon)
        temp_score = predict_all(temp_row)[hazard_choice]
        rows.append({sensitivity_variable: val, "Predicted Risk": temp_score})

    sens_df = pd.DataFrame(rows)
    fig_sens = px.line(sens_df, x=sensitivity_variable, y="Predicted Risk", markers=True, title=f"{hazard_choice} sensitivity to {sensitivity_variable}")
    st.plotly_chart(fig_sens, use_container_width=True)

    if show_input:
        st.markdown("### Model Input Row")
        st.dataframe(scenario_row, use_container_width=True)

# ============================================================
# PAGE: SCENARIO COMPARISON
# ============================================================

elif page == "📈 Scenario Comparison":
    st.markdown("## 📈 Scenario Comparison Lab")
    st.write("Compare baseline district conditions against multiple built-in climate stress scenarios.")

    district = st.selectbox("Choose district for scenario comparison", sorted(df["District"].unique()))
    district_latest = df[df["District"] == district].sort_values("Year").iloc[-1]
    base_values = (
        int(district_latest["Year"]),
        float(district_latest["Avg_Temperature_C"]),
        int(district_latest["Annual_Rainfall_mm"]),
        int(district_latest["AQI"]),
        float(district_latest["Forest_Cover_Percent"]),
        float(district_latest["River_Water_Level_m"]),
        float(district_latest["Carbon_Emission_Metric_Tons_per_Capita"])
    )

    scenarios = [
        "Custom",
        "Heavy rainfall / flood stress",
        "Cyclone-prone coastal stress",
        "Extreme drought / heatwave",
        "Environmental improvement",
        "Worst-case compound hazard"
    ]

    comparison_rows = []
    for preset in scenarios:
        vals = apply_preset(preset, base_values)
        row = make_input_row(district, vals[0], vals[1], vals[2], vals[3], vals[4], vals[5], vals[6])
        preds = predict_all(row)
        for hazard, score in preds.items():
            comparison_rows.append({
                "Scenario": "Baseline" if preset == "Custom" else preset,
                "Hazard": hazard,
                "Risk Score": score
            })

    comp = pd.DataFrame(comparison_rows)
    fig = px.bar(comp, x="Hazard", y="Risk Score", color="Scenario", barmode="group", height=520)
    st.plotly_chart(fig, use_container_width=True)

    pivot = comp.pivot_table(index="Scenario", columns="Hazard", values="Risk Score").reset_index()
    st.dataframe(pivot, use_container_width=True)

# ============================================================
# PAGE: RISK RADAR
# ============================================================

elif page == "🗺️ Multi-Hazard Risk Radar":
    st.markdown("## 🗺️ Multi-Hazard Risk Radar")

    risk_col = st.selectbox("Select risk layer", RISK_COLUMNS)
    top_n = st.slider("Number of top districts", 5, 25, 15)

    top = latest_df.groupby("District")[risk_col].mean().sort_values(ascending=False).reset_index().head(top_n)

    fig = px.bar(
        top,
        x=risk_col,
        y="District",
        orientation="h",
        color=risk_col,
        color_continuous_scale="Turbo",
        title=f"Top {top_n} Districts by {risk_col}"
    )
    fig.update_layout(yaxis={"categoryorder": "total ascending"})
    st.plotly_chart(fig, use_container_width=True)

    fig_map = px.scatter_mapbox(
        latest_df,
        lat="Latitude",
        lon="Longitude",
        color=risk_col,
        size=risk_col,
        hover_name="District",
        zoom=5.5,
        height=520,
        color_continuous_scale="Turbo"
    )
    fig_map.update_layout(mapbox_style="carto-darkmatter", margin={"r":0, "t":0, "l":0, "b":0})
    st.plotly_chart(fig_map, use_container_width=True)

# ============================================================
# PAGE: MODEL LEADERBOARD
# ============================================================

elif page == "🤖 Model Leaderboard":
    st.markdown("## 🤖 Model Leaderboard")
    st.markdown("""
    <div class="warning-card">
    These are real model metrics from the current dataset and engineered risk-index targets.
    No artificial R² replacement or hardcoded accuracy values are used.
    </div>
    """, unsafe_allow_html=True)

    st.dataframe(leaderboard.sort_values(["Hazard", "Test_R2"], ascending=[True, False]), use_container_width=True)

    best_summary = leaderboard.sort_values("Test_R2", ascending=False).groupby("Hazard").head(1)
    fig = px.bar(best_summary, x="Hazard", y="Test_R2", color="Test_R2", color_continuous_scale="Turbo", title="Best Test R² by Hazard")
    st.plotly_chart(fig, use_container_width=True)

# ============================================================
# PAGE: DISTRICT EXPLORER
# ============================================================

elif page == "📍 District Explorer":
    st.markdown("## 📍 District Explorer")

    district = st.selectbox("Select district", sorted(df["District"].unique()))
    ddf = df[df["District"] == district].sort_values("Year")
    latest = ddf.iloc[-1]

    c1, c2, c3, c4, c5 = st.columns(5)
    c1.metric("Flood", f"{latest['Flood_Risk_Index']:.2f}")
    c2.metric("Cyclone", f"{latest['Cyclone_Risk_Index']:.2f}")
    c3.metric("Drought", f"{latest['Drought_Risk_Index']:.2f}")
    c4.metric("Agri Stress", f"{latest['Agricultural_Stress_Index']:.2f}")
    c5.metric("Composite", f"{latest['Composite_Vulnerability_Index']:.2f}")

    fig = px.line(
        ddf,
        x="Year",
        y=RISK_COLUMNS,
        markers=True,
        title=f"Historical risk trends — {district}"
    )
    st.plotly_chart(fig, use_container_width=True)

    st.markdown("### District Records")
    st.dataframe(ddf, use_container_width=True)

# ============================================================
# PAGE: DATASET WORKSPACE
# ============================================================

elif page == "📊 Dataset Workspace":
    st.markdown("## 📊 Dataset Workspace")

    c1, c2 = st.columns([1, 1])
    with c1:
        query = st.text_input("Search district")
    with c2:
        cols_to_show = st.multiselect("Risk columns", RISK_COLUMNS, default=RISK_COLUMNS)

    if query:
        filtered = df[df["District"].str.contains(query, case=False, na=False)]
    else:
        filtered = df

    base_cols = ["District", "Year", "Avg_Temperature_C", "Annual_Rainfall_mm", "AQI", "Forest_Cover_Percent", "River_Water_Level_m", "Carbon_Emission_Metric_Tons_per_Capita"]
    show_cols = base_cols + cols_to_show
    show_cols = [c for c in show_cols if c in filtered.columns]

    st.dataframe(filtered[show_cols], use_container_width=True)

    csv_data = filtered.to_csv(index=False).encode("utf-8")
    st.download_button(
        "Download filtered dataset",
        data=csv_data,
        file_name="multi_hazard_dynamic_dataset.csv",
        mime="text/csv",
        use_container_width=True
    )


Writing /content/app.py



## 5. Syntax Check

This checks whether `app.py` has any Python syntax errors before running Streamlit.


In [ ]:

!python -m py_compile /content/app.py
print("✅ app.py syntax check passed")


✅ app.py syntax check passed



## 6. Patch for Fast Streamlit Loading

This keeps the app lightweight in Colab. It follows the same working style you used earlier.


In [ ]:

!pkill -f streamlit || true

from pathlib import Path

app_path = Path("/content/app.py")
code = app_path.read_text()

# Reduce heavy tree counts for fast Streamlit loading if any larger values exist
code = code.replace("n_estimators=800", "n_estimators=120")
code = code.replace("n_estimators=600", "n_estimators=120")
code = code.replace("n_estimators=500", "n_estimators=120")
code = code.replace("n_estimators=1000", "n_estimators=120")

app_path.write_text(code)

print("✅ app.py patched for faster Streamlit loading")


^C
✅ app.py patched for faster Streamlit loading



## 7. Run Streamlit in Colab

This starts Streamlit on port `8501` and checks the health endpoint.

Expected output:

`ok`


In [ ]:

!pkill -f streamlit || true
!fuser -k 8501/tcp || true

!python -m streamlit run /content/app.py \
  --server.port 8501 \
  --server.address 0.0.0.0 \
  --server.headless true \
  --server.enableCORS false \
  --server.enableXsrfProtection false \
  > /content/streamlit.log 2>&1 &

import time
time.sleep(8)

!curl http://localhost:8501/_stcore/health


^C
ok


## 8. Open the App Inside Colab

Use this instead of Cloudflare if Cloudflare shows a blank page.


In [ ]:

from google.colab import output
output.serve_kernel_port_as_iframe(8501, height=900)


<IPython.core.display.Javascript object>


## 9. Debug Log

Only run this if the app does not open or shows an error.


In [ ]:

!cat /content/streamlit.log | tail -120




2026-05-19 12:05:23.365 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.41.202.252:8501




## What the App Predicts

The app predicts **risk-index scores from 0 to 10**:

| Output | Meaning |
|---|---|
| Flood Risk Index | Flood vulnerability under the selected scenario |
| Cyclone Risk Index | Cyclone/coastal-storm vulnerability |
| Drought Risk Index | Heat, low-rainfall, low-water drought vulnerability |
| Agricultural Stress Index | Climate pressure on agricultural conditions |
| Composite Vulnerability Index | Overall multi-hazard vulnerability |

The app is best described as a **scenario-based multi-hazard risk simulator**, not an official disaster forecasting system.
